# Fresh-process restored model inference

**Inputs:** `AHRV3_WORKSPACE` points at the isolated restored bundle, `AHRV3_RESTORE_RUNS` names its trained checkpoints. **Outputs:** a legal verified action and checkpoint identity for every restored model, `reports/ahr_jepa_v3/restore_inference.json`. **Runtime:** under five minutes for six models. **Resume:** same environment and fresh kernel, Run All. No training. Default Run All only imports definitions unless restore run IDs are supplied.

In [ ]:
from pathlib import Path
import nbformat,os
PROJECT=Path(os.environ.get('AHRV3_WORKSPACE',Path.cwd())).resolve()
while not (PROJECT/'notebooks/00_instance_setup_and_recovery.ipynb').is_file():
    if PROJECT.parent==PROJECT:raise RuntimeError('Run within AHR Go workspace')
    PROJECT=PROJECT.parent
for cell in nbformat.read(PROJECT/'experiments/ahr_jepa_v3/00_shared.ipynb',4).cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(compile(cell.source,'v3-shared:export','exec'),globals())
v3_load_upstream(PROJECT,globals())
for name in ('08_predictive_ahr.ipynb','09_trajectory_learning.ipynb'):
    v3_import_exports(PROJECT/V3_ROOT/name,globals())


In [ ]:
runs=json.loads(os.environ.get('AHRV3_RESTORE_RUNS','[]'))
if runs:v3_verify_original_sources(PROJECT)
rows=[]
for run in runs:
    model,metadata=v3_load_model(PROJECT,run,device='cpu',selection='latest')
    state=GoState(size=9,board=(0,)*81)
    decision=v3_choose_move(state,model.eval(),time_budget_s=5.0,search_config=V3SearchConfig(max_depth=1,max_nodes=16,branch_width=4))
    action=decision['action']
    assert action in legal_actions(state) and verify_go_emission(state,action,play(state,action))
    rows.append({'run_id':run,'checkpoint':metadata,'action':action,'legal':True})
if runs:
    receipt={'status':'PASS','fresh_process':True,'device':'cpu','run_ids':runs,'models':rows,'optimizer_steps_performed':0}
    atomic_json(v3_report_path(PROJECT,'restore_inference.json'),receipt)
    print(json.dumps({'status':'PASS','restored_models':len(rows),'training_steps':0},indent=2))
else:print('INSPECT_ONLY: controller supplies an isolated restored workspace and model list.')